# 01 - Data Cleaning
**Project:** E-Commerce Sales & Customer Analytics

This notebook loads the raw transactional dataset, profiles its data-quality issues,
cleans it with justified, documented decisions, engineers a few analysis-ready features,
and saves the cleaned dataset to `data/cleaned/ecommerce_cleaned.csv`.

**Workflow:** Load -> Inspect -> Profile data quality -> Clean -> Feature engineer -> Save


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

RAW_PATH = '../data/raw/ecommerce_raw.csv'
CLEANED_PATH = '../data/cleaned/ecommerce_cleaned.csv'


## 1. Load raw data

In [2]:
df = pd.read_csv(RAW_PATH)
print("Shape:", df.shape)
df.head()


Shape: (42504, 17)


,Order_ID,Order_Date,Customer_ID,Customer_Name,Product_ID,Product_Name,Category,Sub_Category,Quantity,Sales,Discount,Profit,Region,State,City,Payment_Mode,Shipping_Mode
0,ORD-022056,2023-04-18 00:00:00,CUST-02509,Daniel Reynolds,PROD-00230,Appliances Every Plus,Home & Kitchen,Appliances,2,13126.06,0.15,243.20,West,Maharashtra,Mumbai,Net Banking,Express
1,ORD-005772,11/01/2023,CUST-03733,Mr. Joseph Thompson,PROD-00140,Footwear Economic Plus,Clothing,Footwear,2,6622.50,0.15,136.18,West,Rajasthan,Jaipur,UPI,Standard
2,ORD-009049,2022-10-03 00:00:00,CUST-00972,Jenna Gamble,PROD-00264,Lighting Feel Classic,Home & Kitchen,Lighting,1,5130.10,0.20,-238.48,East,Bihar,Bhagalpur,Net Banking,Standard
3,ORD-007746,2022-06-03 00:00:00,CUST-03143,Kirk Moreno,PROD-00171,Binders Star Max,Office Supplies,Binders,3,3078.05,0.40,-555.71,East,West Bengal,Kolkata,Debit Card,Standard
4,ORD-026616,2023-10-15 00:00:00,CUST-01439,Lisa Nelson,PROD-00237,Decor Quickly Plus,Home & Kitchen,Decor,2,4022.66,0.45,-1021.22,North,Punjab,Amritsar,UPI,Standard


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42504 entries, 0 to 42503
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Order_ID       42504 non-null  object 
 1   Order_Date     42492 non-null  object 
 2   Customer_ID    42504 non-null  object 
 3   Customer_Name  42080 non-null  object 
 4   Product_ID     42504 non-null  object 
 5   Product_Name   42504 non-null  object 
 6   Category       42504 non-null  object 
 7   Sub_Category   42504 non-null  object 
 8   Quantity       42504 non-null  int64  
 9   Sales          42291 non-null  float64
 10  Discount       41867 non-null  float64
 11  Profit         42504 non-null  float64
 12  Region         42504 non-null  object 
 13  State          42504 non-null  object 
 14  City           41651 non-null  object 
 15  Payment_Mode   42078 non-null  object 
 16  Shipping_Mode  42504 non-null  object 
dtypes: float64(3), int64(1), object(13)
memory usage: 

## 2. Data quality profiling

Before touching anything, we quantify exactly what is wrong with the raw file.

In [4]:
print("Missing values per column:")
print(df.isna().sum()[df.isna().sum() > 0])


Missing values per column:
Order_Date        12
Customer_Name    424
Sales            213
Discount         637
City             853
Payment_Mode     426
dtype: int64


In [5]:
print("Fully duplicated rows:", df.duplicated().sum())
dup_order_ids = df[df.duplicated(subset='Order_ID', keep=False)]
print("Rows involved in duplicate Order_IDs:", len(dup_order_ids))
dup_order_ids.sort_values('Order_ID').head(10)


Fully duplicated rows: 396
Rows involved in duplicate Order_IDs: 1014


,Order_ID,Order_Date,Customer_ID,Customer_Name,Product_ID,Product_Name,Category,Sub_Category,Quantity,Sales,Discount,Profit,Region,State,City,Payment_Mode,Shipping_Mode
20746,ORD-000001,2023-09-11 00:00:00,CUST-01019,Jimmy Marks,PROD-00098,Furnishings Hope Elite,Furniture,Furnishings,3,46952.67,0.00,4735.60,West,Rajasthan,Udaipur,Debit Card,Express
7245,ORD-000001,2022-08-11 00:00:00,CUST-01984,Eric Cox,PROD-00253,Storage Son Elite,Home & Kitchen,Storage,1,9245.87,0.10,532.01,South,Tamil Nadu,Madurai,Wallet,Express
13808,ORD-000001,2022-10-22 00:00:00,CUST-03425,Amanda Walters,PROD-00326,Apparel Beat Plus,Sports,Apparel,1,5084.54,0.15,245.97,North,Delhi,Dwarka,Credit Card,Same Day
24816,ORD-000001,09/03/2022,CUST-01909,Shaun Meyer DDS,PROD-00186,Art About Lite,Office Supplies,Art,1,1666.96,0.00,312.13,East,Bihar,Bhagalpur,Debit Card,Standard
36534,ORD-000001,2022-08-11 00:00:00,CUST-01984,Eric Cox,PROD-00253,Storage Son Elite,Home & Kitchen,Storage,1,9245.87,0.10,532.01,South,Tamil Nadu,Madurai,wallet,Express
8122,ORD-000001,2022-01-16 00:00:00,CUST-01118,Erin Smith,PROD-00157,Paper Fund Plus,Office Supplies,Paper,1,1696.73,0.00,302.30,west,Rajasthan,Jaipur,Debit Card,Express
34214,ORD-000001,2023-10-08 00:00:00,CUST-02891,Brian Sosa,PROD-00018,Laptops Television Plus,Electronics,Laptops,1,28978.42,0.10,2140.91,East,Bihar,Patna,UPI,Same Day
2317,ORD-000001,2022-04-17 00:00:00,CUST-01603,Nathan Wilson,PROD-00294,Outdoor Already Plus,Sports,Outdoor,1,2766.12,0.10,336.71,West,Maharashtra,Nagpur,Debit Card,Express
26861,ORD-000114,2023-05-11 00:00:00,CUST-01130,James Rivers,PROD-00299,Team Sports Something Max,Sports,Team Sports,3,15288.78,0.00,3095.51,North,Delhi,Rohini,UPI,Economy
7887,ORD-000114,2023-05-11 00:00:00,CUST-01130,James Rivers,PROD-00299,Team Sports Something Max,Sports,Team Sports,3,15288.78,0.00,3095.51,North,Delhi,Rohini,UPI,Economy


In [6]:
print("Unique Region values (raw):", sorted(df['Region'].dropna().astype(str).unique()))
print()
print("Unique Payment_Mode values (raw):", sorted(df['Payment_Mode'].dropna().astype(str).unique()))


Unique Region values (raw): [' East ', ' North ', ' South ', ' West ', 'EAST', 'East', 'East  ', 'NORTH', 'North', 'North  ', 'SOUTH', 'South', 'South  ', 'WEST', 'West', 'West  ', 'east', 'north', 'south', 'west']

Unique Payment_Mode values (raw): ['Cash on Delivery', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI', 'Wallet', 'cash_on_delivery', 'credit_card', 'debit_card', 'net_banking', 'upi', 'wallet']


In [7]:
print("Quantity summary:")
print(df['Quantity'].describe())
print("\nNegative quantity rows:", (df['Quantity'] < 0).sum())

print("\nDiscount summary:")
print(df['Discount'].describe())
print("Discount > 1 (impossible, >100%):", (df['Discount'] > 1).sum())

print("\nSales == 0 or missing:", ((df['Sales'] == 0) | (df['Sales'].isna())).sum())


Quantity summary:
count    42504.000000
mean         2.049525
std          1.212837
min         -1.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         18.000000
Name: Quantity, dtype: float64

Negative quantity rows: 25

Discount summary:
count    41867.000000
mean         0.106829
std          0.104606
min          0.000000
25%          0.000000
50%          0.100000
75%          0.150000
max          1.500000
Name: Discount, dtype: float64
Discount > 1 (impossible, >100%): 15

Sales == 0 or missing: 233


In [8]:
print("Order_Date raw dtype:", df['Order_Date'].dtype)
print("Sample of differently-formatted dates:")
print(df['Order_Date'].sample(15, random_state=1).tolist())


Order_Date raw dtype: object
Sample of differently-formatted dates:
['2023-04-16 00:00:00', '2023-08-01 00:00:00', '2022-04-18 00:00:00', '2022-08-17 00:00:00', '2022-11-21 00:00:00', '2023-04-06 00:00:00', '2022-10-07 00:00:00', '2023-09-14 00:00:00', '2023-06-24 00:00:00', '2022-11-20 00:00:00', '2022-01-22 00:00:00', '2022-08-18 00:00:00', '2022-01-27 00:00:00', '2022-09-28 00:00:00', '2022/03/15']


### Data quality summary

| Issue | Count | Decision |
|---|---|---|
| Fully duplicated rows | ~396 | Drop (exact double-submission) |
| Duplicate `Order_ID` with different content | ~222 rows | Keep first occurrence, drop rest |
| Missing `Customer_Name` | ~424 | Fill with `'Unknown Customer'` (Customer_ID still valid, so record is still usable) |
| Missing `City` | ~853 | Fill with `'Unknown'` |
| Missing `Payment_Mode` | ~426 | Fill with `'Unknown'` |
| Missing `Discount` | ~637 | Impute with category median discount |
| Missing/zero `Sales` | ~230 | Drop - a sale record with no revenue is not usable for revenue/profit analysis |
| Inconsistent `Region` / `Payment_Mode` casing & whitespace | ~5-8% of rows | Standardize (`strip()`, `title()`, map synonyms) |
| Negative `Quantity` | 25 | Data-entry sign error -> take absolute value |
| `Discount` > 100% | 15 | Impossible -> treat as missing, then impute |
| Multiple date string formats | ~4% | Parse with multiple format attempts |
| Future order dates (2027) | 8 | Impossible for a 2022-2023 dataset -> drop |
| Missing/unparseable dates | 12 | Drop - cannot be used in any time-based analysis |
| Extreme Sales outliers | ~2,308 candidates | **Investigated, not blindly removed** - see below |


## 3. Cleaning

### 3.1 Remove duplicates

In [9]:
before = len(df)
df = df.drop_duplicates()
print(f"Removed {before - len(df)} fully duplicated rows")

before = len(df)
df = df.drop_duplicates(subset='Order_ID', keep='first')
print(f"Removed {before - len(df)} rows with duplicate Order_ID (kept first occurrence)")
print("Shape now:", df.shape)


Removed 396 fully duplicated rows
Removed 114 rows with duplicate Order_ID (kept first occurrence)
Shape now: (41994, 17)


### 3.2 Standardize categorical formatting

In [10]:
df['Region'] = df['Region'].astype(str).str.strip().str.title()

PAYMENT_MAP = {
    'Net Banking': 'Net Banking', 'Upi': 'UPI', 'Debit Card': 'Debit Card',
    'Cash On Delivery': 'Cash on Delivery', 'Credit Card': 'Credit Card', 'Wallet': 'Wallet'
}
df['Payment_Mode'] = df['Payment_Mode'].astype(str).str.replace('_', ' ').str.strip().str.title()
df['Payment_Mode'] = df['Payment_Mode'].replace('Nan', np.nan)
df['Payment_Mode'] = df['Payment_Mode'].map(PAYMENT_MAP).fillna(df['Payment_Mode'])

print(sorted(df['Region'].unique()))
print(sorted(df['Payment_Mode'].dropna().unique()))


['East', 'North', 'South', 'West']
['Cash on Delivery', 'Credit Card', 'Debit Card', 'Net Banking', 'UPI', 'Wallet']


### 3.3 Parse and validate dates

In [11]:
def parse_date(x):
    if pd.isna(x):
        return pd.NaT
    for fmt in ('%Y-%m-%d', '%Y-%m-%d %H:%M:%S', '%d-%m-%Y', '%m/%d/%Y', '%Y/%m/%d'):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(x, errors='coerce')

df['Order_Date'] = df['Order_Date'].apply(parse_date)

before = len(df)
df = df.dropna(subset=['Order_Date'])
print(f"Dropped {before - len(df)} rows with unparseable/missing Order_Date")

before = len(df)
df = df[df['Order_Date'] <= pd.Timestamp('2023-12-31')]
print(f"Dropped {before - len(df)} rows with impossible future Order_Date")


Dropped 12 rows with unparseable/missing Order_Date
Dropped 8 rows with impossible future Order_Date


### 3.4 Handle missing values

In [12]:
df['Customer_Name'] = df['Customer_Name'].fillna('Unknown Customer')
df['City'] = df['City'].fillna('Unknown')
df['Payment_Mode'] = df['Payment_Mode'].fillna('Unknown')

# Discount: impossible values (>100%) are treated as missing, then imputed
df.loc[df['Discount'] > 1, 'Discount'] = np.nan
df['Discount'] = df.groupby('Category')['Discount'].transform(lambda x: x.fillna(x.median()))

# Sales missing or exactly zero cannot be analyzed as revenue -> drop those order lines
before = len(df)
df = df[~((df['Sales'].isna()) | (df['Sales'] == 0))]
print(f"Dropped {before - len(df)} rows with missing/zero Sales")

print(df.isna().sum().sum(), "missing values remaining")


Dropped 230 rows with missing/zero Sales
0 missing values remaining


### 3.5 Fix invalid numeric values

In [13]:
neg_qty = (df['Quantity'] < 0).sum()
df.loc[df['Quantity'] < 0, 'Quantity'] = df.loc[df['Quantity'] < 0, 'Quantity'].abs()
print(f"Corrected {neg_qty} negative Quantity values via absolute value (sign-entry error)")


Corrected 25 negative Quantity values via absolute value (sign-entry error)


### 3.6 Outlier investigation (Sales)

We do **not** blindly delete statistical outliers. Instead we check whether a high-Sales
row is explained by a correspondingly high `Quantity` (a genuine bulk/corporate order) or
whether Sales is extreme while Quantity stays low (a likely data error).

In [14]:
Q1, Q3 = df['Sales'].quantile([0.25, 0.75])
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR   # wide bound: only flag extreme outliers, not every high-value order
candidates = df[df['Sales'] > upper_bound]
print(f"Extreme Sales outlier candidates: {len(candidates)}")

# Genuine bulk orders: Quantity is also proportionally high
genuine_bulk = candidates[candidates['Quantity'] > 2]
suspicious = candidates[candidates['Quantity'] <= 2]
print(f"  -> Kept as genuine bulk/corporate orders (high Sales AND high Quantity): {len(genuine_bulk)}")
print(f"  -> Removed as likely data errors (high Sales, low Quantity): {len(suspicious)}")

df = df.drop(suspicious.index)
print("Shape after outlier handling:", df.shape)


Extreme Sales outlier candidates: 2308
  -> Kept as genuine bulk/corporate orders (high Sales AND high Quantity): 1826
  -> Removed as likely data errors (high Sales, low Quantity): 482
Shape after outlier handling: (41262, 17)


## 4. Feature engineering

In [15]:
df['Year'] = df['Order_Date'].dt.year
df['Month'] = df['Order_Date'].dt.month
df['Month_Name'] = df['Order_Date'].dt.strftime('%b')
df['Quarter'] = df['Order_Date'].dt.quarter
df['Day_of_Week'] = df['Order_Date'].dt.day_name()
df['Profit_Margin'] = (df['Profit'] / df['Sales']).round(4)
df['Unit_Price'] = (df['Sales'] / df['Quantity']).round(2)

df[['Order_Date','Year','Month','Month_Name','Quarter','Day_of_Week','Sales','Profit','Profit_Margin','Unit_Price']].head()


,Order_Date,Year,Month,Month_Name,Quarter,Day_of_Week,Sales,Profit,Profit_Margin,Unit_Price
0,2023-04-18,2023,4,Apr,2,Tuesday,13126.06,243.20,0.0185,6563.03
1,2023-11-01,2023,11,Nov,4,Wednesday,6622.50,136.18,0.0206,3311.25
2,2022-10-03,2022,10,Oct,4,Monday,5130.10,-238.48,-0.0465,5130.10
3,2022-06-03,2022,6,Jun,2,Friday,3078.05,-555.71,-0.1805,1026.02
4,2023-10-15,2023,10,Oct,4,Sunday,4022.66,-1021.22,-0.2539,2011.33


## 5. Final validation

In [16]:
print("Final shape:", df.shape)
print("\nRemaining nulls:\n", df.isna().sum()[df.isna().sum() > 0])
print("\nDuplicate Order_IDs remaining:", df['Order_ID'].duplicated().sum())
print("\nQuantity < 0:", (df['Quantity'] < 0).sum())
print("Discount > 1:", (df['Discount'] > 1).sum())
print("Sales <= 0:", (df['Sales'] <= 0).sum())
print("\nDate range:", df['Order_Date'].min(), "to", df['Order_Date'].max())
df.dtypes


Final shape: (41262, 24)

Remaining nulls:
 Series([], dtype: int64)

Duplicate Order_IDs remaining: 0

Quantity < 0: 0
Discount > 1: 0
Sales <= 0: 0

Date range: 2022-01-01 00:00:00 to 2023-12-28 00:00:00


Order_ID                 object
Order_Date       datetime64[ns]
Customer_ID              object
Customer_Name            object
Product_ID               object
Product_Name             object
Category                 object
Sub_Category             object
Quantity                  int64
Sales                   float64
Discount                float64
Profit                  float64
Region                   object
State                    object
City                     object
Payment_Mode             object
Shipping_Mode            object
Year                      int32
Month                     int32
Month_Name               object
Quarter                   int32
Day_of_Week              object
Profit_Margin           float64
Unit_Price              float64
dtype: object

## 6. Save cleaned dataset

In [17]:
import os
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv(CLEANED_PATH, index=False)
print(f"Saved cleaned dataset with shape {df.shape} to {CLEANED_PATH}")


Saved cleaned dataset with shape (41262, 24) to ../data/cleaned/ecommerce_cleaned.csv


## Summary

Starting from **42,504** raw rows, cleaning removed exact duplicates, duplicate
`Order_ID`s, unparseable/impossible dates, unusable (missing/zero) Sales rows, and a small
number of statistically extreme *and* logically inconsistent Sales outliers, while
correcting fixable errors (negative quantities, inconsistent text formatting, missing
categorical values) rather than discarding them. The result is a clean, analysis-ready
dataset used in the rest of this project.
